SLMPart2

### Setting Up The Foundation

In [ ]:
!nvidia-smi

In [ ]:
# Load Hugging Face Token from shared environment
# 
from dotenv import load_dotenv
import os
load_dotenv('../shared/.env')
key = os.getenv('HF_TOKEN')
os.environ["HF_TOKEN"] = key

In [ ]:
# install required libraries  
# This is not on the image so we need to install


!pip install accelerate>=0.26.0 --quiet
print("Libraries installed!")

In [ ]:
import torch # Base PyTorch library, Gemma runs on this
from transformers import AutoModelForCausalLM, AutoTokenizer # For loading Gemma from Hugging Face

# LlamaIndex specific imports
from pypdf import PdfReader
from llama_index.core import Document, VectorStoreIndex, Settings # This is a central place to configure defaults in LlamaIndex
from llama_index.core.node_parser import SentenceSplitter #Breaks pages into chunks
from llama_index.llms.huggingface import HuggingFaceLLM # A wrapper to make our Hugging Face Gemma model compatible with LlamaIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding # A wrapper for using Hugging Face embedding models

In [ ]:
import time

In [ ]:
# --- Load Gemma 2 2B ---
local_model_path = "/home/jovyan/shared/models/google--gemma-2-2b-it"

tokenizer_gemma = AutoTokenizer.from_pretrained(local_model_path)
t0 = time.time()
model_gemma = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
#Settings.llm = model_gemma
print(f"Gemmaload took {time.time()-t0:.2f}s")
print(f"Gemma 2 2B loaded")

In [ ]:
# =====  Now trty gguf
from llama_cpp import Llama

read_path = "/home/jovyan/shared/models/gguf/gemma-2-2b-it.q4_k_m.gguf"

t0 = time.time()
llmGemG = Llama(model_path=read_path, n_gpu_layers=-1, verbose =False)
print(f"GGUF load took {time.time()-t0:.2f}s")

In [ ]:
prompt = "Write a short story about a time-traveling cat"

In [ ]:

!nvidia-smi

In [ ]:
# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens, temperature):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
        temperature=temperature,      # Control the randomness (creativity vs focus)
        do_sample=True,               # Enable sampling (needed for temperature)
        #pad_token_id=tokenizer.eos_token_id # Prevent warnings about padding token
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [ ]:
# Define a function to handle text generation and timing
def generate_text_gguf(model, prompt, max_new_tokens, temperature):

    start_time = time.time() # Record the time before generation starts

    
# Create the messages list (OpenAI-compatible format)
    messages = [
           {"role": "user", "content": prompt}
    ]

# Generate a response using create_chat_completion
    response = model.create_chat_completion(
       temperature = temperature,
       messages = messages,
       max_tokens = max_new_tokens
    )

# Extract and print the response
    print("Response:")
    print(response["choices"][0]["message"]["content"])
    # Generate text using the model
    output =  response["choices"][0]["message"]["content"]
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    completion_tokens = response["usage"]["completion_tokens"]
    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = len(output) / completion_tokens
    print(f"Generated {completion_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens per sec)")

   
    # Return the text and speed for potential later use
    return output, tokens_per_sec

In [ ]:
# low temperature
_ = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=0.2)

In [ ]:
# low temperature
_ = generate_text_gguf(llmGemG, prompt, max_new_tokens=500, temperature=.2)

In [ ]:
# low temperature
_ = generate_text_gguf(llmGemG, prompt, max_new_tokens=500, temperature=1.0)

**RAG**

**RAG**

In [ ]:
# Define a system prompt to guide Gemma's behavior for our Q&A task
# This helps instruct the model to focus on the provided context
system_prompt = "You are a helpful and precise Q&A assistant. Your primary goal is to answer questions based *only* on the contextual information provided to you. If the answer cannot be found within the provided context, please state clearly 'The provided context does not contain the answer to this question.'"

# Create the LlamaIndex SLM wrapper for our loaded Gemma model
llm_gemma = HuggingFaceLLM(
    model=model_gemma,                     # The loaded Hugging Face model object
    tokenizer=tokenizer_gemma,             # The loaded Hugging Face tokenizer object
    context_window=8192,                   # Gemma 2 2B's maximum context window size in tokens
    max_new_tokens=500,                    # Default maximum number of tokens for generated answers
    generate_kwargs={"temperature": 0.2, "do_sample": True}, # Default generation parameters (low temp for factual Q&A)
    system_prompt=system_prompt,           # The system prompt defined above
    device_map="auto"                      # Ensure LlamaIndex knows the model is on the GPU
)

# Set this configured SLM as the default for all LlamaIndex operations
Settings.llm = llm_gemma
print("\n Gemma 2 2B SLM wrapped and set as default in LlamaIndex Settings.")

In [ ]:
# Define the Hugging Face ID for a good, efficient embedding model
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"

# Create the LlamaIndex embedding model object using the HuggingFaceEmbedding wrapper
embed_model = HuggingFaceEmbedding(model_name=embed_model_id)

# Set this embedding model as the default for all LlamaIndex operations
Settings.embed_model = embed_model

# We can also globally set a default chunk size that LlamaIndex will use
# when it splits our document into manageable pieces. 512 is a common starting point.
Settings.chunk_size = 512

### Building the Q&A Pipeline

---- Need to put file in same  directory as notebook -----

In [ ]:
reader = PdfReader("Artificial intelligence - Wikipedia.pdf")
docs = [
    Document(text=page.extract_text() or "", metadata={"source": "paper.pdf", "page": i + 1})
    for i, page in enumerate(reader.pages)
]
docs = [d for d in docs if d.text.strip()]

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(docs)
print(f"{len(docs)} pages -> {len(nodes)} chunks")

#Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

index = VectorStoreIndex(nodes)
# This command tells LlamaIndex to take our document,
# process them (chunk, embed using Settings.embed_model and Settings.chunk_size),
# and build a searchable vector index.
print("Index built successfully!")

In [ ]:
print('Files in current directory:')
!ls -F
print('\n(Please ensure the PDF filename above matches exactly with the `pdf_filename` variable in the previous cell.)')

In [ ]:
%from llama_index.core import VectorStoreIndex

print("\nBuilding the VectorStoreIndex from the loaded document(s)...")



# This command tells LlamaIndex to take our list of 'documents',
# process them (chunk, embed using Settings.embed_model and Settings.chunk_size),
# and build a searchable vector index.
# This step can take a little while for a 33-page PDF, as it's embedding all the chunks.
index = VectorStoreIndex.from_documents(documents)
print("Index built successfully!")

In [ ]:
query_engine = index.as_query_engine()
print("Query engine created and ready to answer questions.")

### Testing the System

In [ ]:
question1 = "What are some of the techniques used in AI research?"
print("Question 1:", question1)

# Send the question to the query engine
response1 = query_engine.query(question1)

print("\nAnswer 1:", response1)

In [ ]:
question2 = "What is the difference between AI and AGI?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

In [ ]:
question2 = "What is the difference between AI and Super Intelligence?"
print("Question 2:", question2)
response2 = query_engine.query(question2)
print("\nAnswer 2:", response2)

In [ ]:
question3 = "Can you explain NLP to me like I'm 5? Make it detailed"
print("Question 3:", question3)
response3 = query_engine.query(question3)
print("\nAnswer 3:", response3)

### Adding Conversational Capabilities

In [ ]:
    # .as_chat_engine() creates an engine suitable for conversation.
    # It typically uses a ChatMemoryBuffer by default to store conversation history.

    # 'chat_mode="condense_question"' is a good default for RAG:
    # It takes the new question and recent chat history, condenses them into a
    # standalone question, and then queries the index with that improved question.
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    verbose=True # Set to True to see some of the internal workings, like the condensed question
)
print("Chat engine created successfully! It's ready for a conversation.")

In [ ]:

# First question
q1 = "Please list six current, real-world applications of artificial intelligence, numbering them 1-6."
print("User:", q1)
r1 = chat_engine.chat(q1)
print("Assistant:", r1)

In [ ]:
# Next follow up
q2 = "Can you elaborate a little more on 5?"
print("User:", q2)
r2 = chat_engine.chat(q2)
print("Assistant:", r2)

In [ ]:
# to clear conversational history
chat_engine.reset()

****-----***

Need to add example using gguf